# evy Quickstart

A simple guide to computing EVI zonal statistics using Google Earth Engine.

In [ ]:
import logging

import evy

logging.basicConfig(level=logging.INFO)

## Basic Usage

The simplest call - monthly EVI for San Marino governorates over the past year:

In [ ]:
df = evy.zonal_stats("SMR")
df.head()

## Choose Data Source

The default data source is MODIS, with 250m resolution. By default, the zonal statistics are masked to cropland areas using the Dynamic World land cover classification. You can disable this behavior by setting `mask_cropland=False`.

In [ ]:
df_modis = evy.zonal_stats(
    "SMR",
    source="modis",
    start_date="2024-01-01",
    end_date="2024-12-31",
    freq=evy.MONTHLY,  # or freq = "ME"
    stats=["mean", "std", "min", "max"],
)

df_modis.head()

Or you can use Sentinel-2 for higher resolution with 10m resolution.

In [ ]:
df_s2 = evy.zonal_stats(
    "SMR",
    source="sentinel2",
    start_date="2024-01-01",
    end_date="2024-12-31",
    freq=evy.MONTHLY,
)

df_s2.head()

## Custom Boundaries

Use your own shapefile or GeoDataFrame:

In [ ]:
gdf = evy.get_boundaries("SMR", admin_level=1)

df_smr = evy.zonal_stats(
    gdf,
    source="modis",
    start_date="2024-10-01",
    end_date="2024-12-31",
    freq=evy.MONTHLY,
    include_geometry=True,
    export_to_drive=True,
)

## Large Jobs - Export to Google Drive

For queries spanning many years, export to Drive instead of downloading directly:

In [ ]:
# This starts an async export task
# task_id = evy.zonal_stats(
#     'SMR',
#     start_date='2010-01-01',
#     end_date='2024-12-31',
#     freq=evy.YEARLY,
#     mask_cropland=True,
#     export_to_drive=True,
#     drive_folder='evy_exports',
# )
#
# print(f"Export started! Task ID: {task_id}")
# print("Check Google Drive for results when complete.")

# Check task status
# status = evy.check_task_status(task_id)
# print(status)

## Available Collections

See what data sources are available:

In [ ]:
evy.list_collections()

## Phenology Extraction (SOS, MOS, EOS)

Extract crop seasonality metrics - Start of Season (SOS), Middle of Season (MOS), and End of Season (EOS):

In [ ]:
phenology = evy.calculate_phenology(df_modis, value_col="mean")
phenology.head()

Calculate the phenology metrics per governorate

In [ ]:
phenology_by_region = evy.calculate_phenology(
    df_modis, value_col="mean", group_col="shapeName"
)
phenology_by_region.head(12)

Filter data to growing season only

In [ ]:
df_growing = evy.filter_growing_season(df_modis, start_month=2, end_month=6)
df_growing.head()

## Visualizations

Built-in interactive charts using Altair:

In [ ]:
chart = evy.plot_seasonality(phenology)
chart

In [ ]:
chart_regions = evy.plot_seasonality_by_region(
    phenology_by_region, region_col="shapeName"
)
chart_regions